In [ ]:
import os
import openai
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
import requests


In [ ]:
deepseek_api = os.environ["DEEPSEEK_API_KEY"]

In [ ]:
from openai import OpenAI

client = OpenAI(api_key=deepseek_api, base_url="https://api.deepseek.com")

response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": "You are a helpful assistant"},
        {"role": "user", "content": "Hello"},
    ],
    stream=False,
    max_tokens=5
)

print(response.choices[0].message.content)

In [ ]:
url = "https://api.deepseek.com/chat/completions"
headers = {
    "Content-Type": "application/json",
    "Authorization": f"Bearer {deepseek_api}"  # Or replace with your API key directly
}

In [ ]:
def classify_snippet_deepseek(snippet, index):
    print(f"Processing index {index}: {snippet[:50]}...")  # Log the snippet

    payload = {
        "model": "deepseek-chat",
        "messages": [
            {"role": "system", "content": "You are a helpful software engineer that categorizes text into one of the following categories: code (code logic), testing (testing code), license (license information), docs (documentation), dicts (dictionary and other type of data carriers). Respond with a single word only."},
            {"role": "user", "content": f"Classify the following snippet: {snippet}"}
        ],
        "max_tokens": 3,
        "temperature": 0
    }
    response = requests.post(url, headers=headers, json=payload)
    

    try:
        response = requests.post(url, headers=headers, json=payload)
        response.raise_for_status()
        category = response.json()["choices"][0]["message"]["content"].strip()
        print(f"Index {index} processed: Category -> {category}")  # Log the result
        return category
    except requests.exceptions.RequestException as e:
        print(f"Error at index {index}: {e}")
        return "error"

In [ ]:
def process_dataframe(df, batch_size=50):
    categories = []
    for i in range(0, len(df), batch_size):
        batch = df['extracted_snippet_full'].iloc[i:i+batch_size]
        print(f"\nProcessing batch {i // batch_size + 1} of {len(df) // batch_size + 1}...\n")
        for index, snippet in batch.items():
            category = classify_snippet_deepseek(snippet, index)
            if category:
                categories.append(category)
            else:
                categories.append("error")  # Handle errors gracefully
            time.sleep(1)  # Rate limit to avoid API overload (adjust as needed)
    df['category_better_prompt_deepseek'] = categories
    return df

In [ ]:
def process_csvs_in_directory(input_dir, output_dir, batch_size=50):
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    # Walk through the directory and find all CSV files
    for root, _, files in os.walk(input_dir):
        for file in files:
            if file.endswith('.csv'):
                input_file_path = os.path.join(root, file)
                output_file_path = os.path.join(output_dir, file)
                
                print(f"\nProcessing file: {input_file_path}")
                
                # Load the CSV into a DataFrame
                df = pd.read_csv(input_file_path)
                
                # Check if the required column exists
                if 'extracted_snippet_full' not in df.columns:
                    print(f"Skipped {file}: 'extracted_snippet_full' column not found.")
                    continue
                
                # Process the DataFrame
                processed_df = process_dataframe(df, batch_size=batch_size)
                
                # Save the updated DataFrame to the output directory
                processed_df.to_csv(output_file_path, index=False)
                print(f"Saved processed file to: {output_file_path}")

In [ ]:
df = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp2_extraction_results/llama_extractions/regular_llama_extraction/regular_llama_extraction/repo_path_size_copies_contents_000000000000.csv")

In [ ]:
df.head()

In [ ]:
input_dir = "/Users/anonymous_user/Downloads/experiment_results/exp2_extraction_results/llama_extractions/regular_llama_extraction/regular_llama_extraction"
output_dir = "/Users/anonymous_user/Downloads/experiment_results/exp3_categorization_results/LLaMa/reg_llama_categorization"

In [ ]:
process_csvs_in_directory(input_dir, output_dir)

In [ ]:
df = pd.read_csv("/Users/anonymous_user/Downloads/experiment_results/exp3_categorization_results/LLaMa/reg_llama_categorization/repo_path_size_copies_contents_000000000018.csv")

In [ ]:
df.head()